In [2]:
import pandas as pd
import json

train_path = "../verl/examples/data_preprocess/output/train.parquet"
test_path  = "../verl/examples/data_preprocess/output/test.parquet"

In [3]:
train_df = pd.read_parquet(train_path)
test_df  = pd.read_parquet(test_path)

def get_user_question(prompt_obj):
    # prompt 是 list[dict]，找最后一个 role=user
    if isinstance(prompt_obj, list):
        for msg in reversed(prompt_obj):
            if isinstance(msg, dict) and msg.get("role") == "user":
                return msg.get("content", "")
        # fallback
        if len(prompt_obj) and isinstance(prompt_obj[-1], dict):
            return prompt_obj[-1].get("content", "")
    return str(prompt_obj)

def get_cot_answer(extra_info_obj):
    if isinstance(extra_info_obj, dict):
        return extra_info_obj.get("answer", "")
    return ""

def to_chat_example(row):
    q = get_user_question(row["prompt"])
    a = get_cot_answer(row["extra_info"])
    return {
        "messages": [
            {"role": "user", "content": q},
            {"role": "assistant", "content": a}
        ]
    }

# 先小样本跑通（比如 500）
train_examples = [to_chat_example(train_df.iloc[i]) for i in range(min(500, len(train_df)))]
val_examples   = [to_chat_example(test_df.iloc[i])  for i in range(min(200, len(test_df)))]

with open("./sft_train.jsonl", "w", encoding="utf-8") as f:
    for ex in train_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

with open("./sft_val.jsonl", "w", encoding="utf-8") as f:
    for ex in val_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print("Saved:", len(train_examples), len(val_examples))

Saved: 500 200


In [ ]:
配置 LoRA（Rank=16）并加载模型

In [4]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else None,
    device_map="auto" if torch.cuda.is_available() else None
)

lora_config = LoraConfig(
    r=16,                # Rank=16 就在这里
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # Qwen 常用 target_modules（如果你后面报找不到模块，再微调）
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj", "gate_proj"]
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 10,092,544 || all params: 761,724,928 || trainable%: 1.3250


In [13]:
!pip install trl

   ---------------------------------------- 0.0/528.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/528.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/528.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/528.8 kB ? eta -:--:--
   ------------------- -------------------- 262.1/528.8 kB ? eta -:--:--
   ---------------------------------------- 528.8/528.8 kB 897.7 kB/s  0:00:01


In [ ]:
把 chat 数据喂给 SFTTrainer（CoT 格式就在 assistant 的内容里）

In [16]:
!pip install --upgrade transformers

In [17]:
import transformers
print(transformers.__version__)

5.2.0


In [5]:

from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig  # 新版 TRL 常用

train_raw = load_dataset("json", data_files="./sft_train.jsonl")["train"]
val_raw   = load_dataset("json", data_files="./sft_val.jsonl")["train"]

def formatting_func(example):
    # 把 messages 套用 Qwen chat template，生成训练文本
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

# 用 SFTConfig（更适配新 SFTTrainer）
sft_args = SFTConfig(
    output_dir="./lora_qwen3_gsm8k_cot",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    eval_strategy="steps",   # 你 transformers 里大概率要用这个名字
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    max_length=512,      # 你MX250建议先 512，别上来 1024
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_raw,
    eval_dataset=val_raw,
    processing_class=tokenizer,   # ✅ 这里替代 tokenizer=
    formatting_func=formatting_func
)

trainer.train()

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

ValueError: Your setup doesn't support bf16/gpu. You need to assign use_cpu if you want to train the model on CPU.

In [6]:
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig  # 新版 TRL 常用

train_raw = load_dataset("json", data_files="./sft_train.jsonl")["train"]
val_raw   = load_dataset("json", data_files="./sft_val.jsonl")["train"]

def formatting_func(example):
    # 把 messages 套用 Qwen chat template，生成训练文本
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

# 用 SFTConfig（更适配新 SFTTrainer）
sft_args = SFTConfig(
    output_dir="./lora_qwen3_gsm8k_cot",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    use_cpu=True,   # ✅ 关键：CPU训练要显式声明
    fp16=False,     # ✅
    bf16=False,     # ✅
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_raw,
    eval_dataset=val_raw,
    processing_class=tokenizer,   # ✅ 这里替代 tokenizer=
    formatting_func=formatting_func
)

trainer.train()


Applying formatting function to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Applying formatting function to eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


KeyboardInterrupt: 

In [23]:
SFTTrainer??

Init signature:
SFTTrainer(
    model: 'str | PreTrainedModel | PeftModel',
    args: trl.trainer.sft_config.SFTConfig | transformers.training_args.TrainingArguments | None = None,
    data_collator: collections.abc.Callable[[list[typing.Any]], dict[str, typing.Any]] | None = None,
    train_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | None = None,
    eval_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | dict[str, datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset] | None = None,
    processing_class: transformers.tokenization_utils_base.PreTrainedTokenizerBase | transformers.processing_utils.ProcessorMixin | None = None,
    compute_loss_func: collections.abc.Callable | None = None,
    compute_metrics: collections.abc.Callable[[transformers.trainer_utils.EvalPrediction], dict] | None = None,
    callbacks: list[transformers.trainer_callback.TrainerCallback] | None = None,
    optimi

In [26]:
import torch
print("cuda available:", torch.cuda.is_available())
print("torch cuda:", torch.version.cuda)

cuda available: False
torch cuda: None


In [ ]:
保存 LoRA adapter（训练结果）

In [ ]:
trainer.model.save_pretrained("./lora_qwen3_gsm8k_cot/adapter")
tokenizer.save_pretrained("./lora_qwen3_gsm8k_cot/tokenizer")
print("Saved LoRA adapter.")

In [ ]:
快速验证（推理测试）

In [ ]:
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
lora_model = PeftModel.from_pretrained(base, "./lora_qwen3_gsm8k_cot/adapter")
lora_model.eval()

q = "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"
messages = [{"role": "user", "content": q}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer([text], return_tensors="pt").to(lora_model.device)

with torch.no_grad():
    out = lora_model.generate(**inputs, max_new_tokens=200, do_sample=False)

print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))